In [1]:
!pip install -q wandb

In [2]:
import random
from pathlib import Path
import numpy as np
import pandas as pd
import torch


seed = 67
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
root = Path(
    "/kaggle/input/datasets/awsaf49/asvpoof-2019-dataset/LA/LA"
)

protocol_dir = root / "ASVspoof2019_LA_cm_protocols"


def load_protocol(protocol_file, audio_dir):
    df = pd.read_csv(protocol_file, sep=r"\s+", header=None)
    df = df[[1, 4]].copy()
    df.columns = ["file_id", "label"]
    df["target"] = (df["label"] == "bonafide").astype(int)
    df["path"] = df["file_id"].apply(
        lambda name: audio_dir / f"{name}.flac"
    )

    return df[["file_id", "target", "path"]]


train_df = load_protocol(
    protocol_dir / "ASVspoof2019.LA.cm.train.trn.txt",
    root / "ASVspoof2019_LA_train" / "flac",
)

dev_df = load_protocol(
    protocol_dir / "ASVspoof2019.LA.cm.dev.trl.txt",
    root / "ASVspoof2019_LA_dev" / "flac",
)

eval_df = load_protocol(
    protocol_dir / "ASVspoof2019.LA.cm.eval.trl.txt",
    root / "ASVspoof2019_LA_eval" / "flac",
)

In [4]:
import soundfile as sf
import torchaudio
import torch.nn.functional as F

from torch.utils.data import Dataset


sample_rate = 16000

n_fft = 512
win_len = 320
hop_len = 160
frames = 750
audio_len = int(sample_rate * 7.5)


lfcc = torchaudio.transforms.LFCC(
    sample_rate=sample_rate,
    n_filter=20,
    n_lfcc=20,
    log_lf=True,
    speckwargs={
        "n_fft": n_fft,
        "win_length": win_len,
        "hop_length": hop_len,
        "window_fn": torch.hann_window,
        "power": 2.0,
    },
)


class AudioDataset(Dataset):
    def __init__(self, table, train=False):
        self.table = table.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.table)

    def __getitem__(self, index):
        row = self.table.iloc[index]

        audio, _ = sf.read(row["path"], dtype="float32")
        audio = torch.from_numpy(audio)

        if audio.ndim == 2:
            audio = audio.mean(dim=1)

        if len(audio) < audio_len:
            audio = F.pad(audio, (0, audio_len - len(audio)))

        elif len(audio) > audio_len:
            if self.train:
                start = torch.randint(
                    0,
                    len(audio) - audio_len + 1,
                    (1,),
                ).item()
            else:
                start = 0

            audio = audio[start:start + audio_len]

        feat = lfcc(audio)

        delta = torchaudio.functional.compute_deltas(feat)
        delta2 = torchaudio.functional.compute_deltas(delta)

        feat = torch.cat([feat, delta, delta2], dim=0)
        feat = feat[:, :frames]

        return (
            feat.unsqueeze(0),
            int(row["target"]),
            row["file_id"],
        )


train_data = AudioDataset(train_df, train=True)
eval_data = AudioDataset(eval_df)

In [5]:
train_data = AudioDataset(train_df, train=True)
dev_data = AudioDataset(dev_df)
eval_data = AudioDataset(eval_df)

загружаем батчами, train перемешиваем, dev и eval оставляем в исходном порядке

In [6]:
from torch.utils.data import DataLoader


batch_size = 32

train_loader = DataLoader(
    train_data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

eval_loader = DataLoader(
    eval_data,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

моделька

In [7]:
import torch.nn as nn


class MFM(nn.Module):
    def forward(self, x):
        first, second = torch.chunk(x, 2, dim=1)
        return torch.maximum(first, second)


class LCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, padding=2),
            MFM(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 96, kernel_size=3, padding=1),
            MFM(),
            nn.MaxPool2d(2),
            nn.BatchNorm2d(48),

            nn.Conv2d(48, 96, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(48),

            nn.Conv2d(48, 128, kernel_size=3, padding=1),
            MFM(),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(64),

            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            MFM(),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 64, kernel_size=1),
            MFM(),
            nn.BatchNorm2d(32),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            MFM(),
            nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 3 * 46, 160),
            MFM(),
            nn.Dropout(0.75),
            nn.BatchNorm1d(80),
            nn.Linear(80, 2),
        )

    def forward(self, x):
        x = self.conv(x)
        return self.fc(x)

проверим на одном батче

In [8]:
small_df = pd.concat([
    train_df[train_df["target"] == 0].iloc[:8],
    train_df[train_df["target"] == 1].iloc[:8],
]).reset_index(drop=True)

small_data = AudioDataset(small_df, train=True)

small_loader = DataLoader(
    small_data,
    batch_size=16,
    shuffle=False,
    num_workers=0,
)

x, y, _ = next(iter(small_loader))

x = x.to(device)
y = y.to(device)

check_model = LCNN().to(device)
check_opt = torch.optim.Adam(
    check_model.parameters(),
    lr=1e-3,
)

check_criterion = nn.CrossEntropyLoss()

for step in range(200):
    check_model.train()

    check_opt.zero_grad()

    logits = check_model(x)
    loss = check_criterion(logits, y)

    loss.backward()
    check_opt.step()

check_model.eval()

with torch.no_grad():
    logits = check_model(x)

    final_loss = check_criterion(logits, y).item()
    final_acc = (
        logits.argmax(dim=1) == y
    ).float().mean().item()

print("final loss:", round(final_loss, 4))
print("final accuracy:", round(final_acc, 4))

del check_model, check_opt, x, y, logits, loss
torch.cuda.empty_cache()

final loss: 0.0041
final accuracy: 1.0


In [14]:
def compute_eer(target_scores, nontarget_scores):
    all_scores = np.concatenate([
        target_scores,
        nontarget_scores,
    ])

    labels = np.concatenate([
        np.ones(len(target_scores)),
        np.zeros(len(nontarget_scores)),
    ])

    order = np.argsort(all_scores, kind="mergesort")

    labels = labels[order]
    all_scores = all_scores[order]

    target_errors = np.cumsum(labels)

    nontarget_errors = len(nontarget_scores) - (
        np.arange(1, len(all_scores) + 1) - target_errors
    )

    frr = np.concatenate([
        [0],
        target_errors / len(target_scores),
    ])

    far = np.concatenate([
        [1],
        nontarget_errors / len(nontarget_scores),
    ])

    thresholds = np.concatenate([
        [all_scores[0] - 0.001],
        all_scores,
    ])

    index = np.argmin(np.abs(frr - far))
    eer = (frr[index] + far[index]) / 2

    return eer, thresholds[index]

In [15]:
from tqdm.auto import tqdm


def train_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    total = 0

    for x, y, _ in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(y)
        total += len(y)

    return total_loss / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0
    total = 0

    all_scores = []
    all_targets = []
    all_names = []

    for x, y, names in tqdm(loader, leave=False):
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        logits = model(x)
        loss = criterion(logits, y)

        total_loss += loss.item() * len(y)
        total += len(y)

        scores = logits[:, 1] - logits[:, 0]

        all_scores.append(scores.cpu().numpy())
        all_targets.append(y.cpu().numpy())
        all_names.extend(names)

    all_scores = np.concatenate(all_scores)
    all_targets = np.concatenate(all_targets)

    bona_scores = all_scores[all_targets == 1]
    spoof_scores = all_scores[all_targets == 0]

    eer, _ = compute_eer(
        bona_scores,
        spoof_scores,
    )

    return (
        total_loss / total,
        eer * 100,
        all_scores,
        all_names,
    )

обучаем модель и сохраняем логи в WandB

In [16]:
import os
import time
import wandb

from kaggle_secrets import UserSecretsClient


os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret(
    "WANDB_API_KEY"
)

wandb.login()

True

In [17]:
config = {
    "model": "LCNN",
    "input": "LFCC",
    "features": 60,
    "n_fft": n_fft,
    "win_len": win_len,
    "hop_len": hop_len,
    "frames": frames,
    "balanced_batches": False,
    "batch_size": batch_size,
    "lr": 3e-4,
    "epochs": 12,
    "seed": seed,
}

run = wandb.init(
    project="ml_summer",
    entity="komarvalera-hse-university",
    name="lcnn_5",
    config=config,
)

model = LCNN().to(device)
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["lr"],
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=10,
    gamma=0.5,
)

best_eer = float("inf")
best_epoch = 0
best_path = Path("/kaggle/working/best_model.pt")

history = []

for epoch in range(1, config["epochs"] + 1):
    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
    )

    eval_loss, eval_eer, _, _ = evaluate(
        model,
        eval_loader,
        criterion,
    )

    if eval_eer < best_eer:
        best_eer = eval_eer
        best_epoch = epoch

        torch.save(
            model.state_dict(),
            best_path,
        )

    row = {
        "epoch": epoch,
        "train_loss": train_loss,
        "eval_loss": eval_loss,
        "eval_eer": eval_eer,
    }

    history.append(row)
    run.log(row)

    print(
        f"epoch {epoch:02d} | "
        f"train loss {train_loss:.4f} | "
        f"eval loss {eval_loss:.4f} | "
        f"eval EER {eval_eer:.3f}%"
    )

    scheduler.step()

history_df = pd.DataFrame(history)

history_df.to_csv(
    "/kaggle/working/history.csv",
    index=False,
)

run.summary["best_eer"] = best_eer
run.summary["best_epoch"] = best_epoch

run.finish()

print("best epoch:", best_epoch)
print("best eval EER:", round(best_eer, 3))

  0%|          | 0/793 [00:00<?, ?it/s]

  0%|          | 0/2227 [00:00<?, ?it/s]

epoch 01 | train loss 0.1559 | eval loss 0.6935 | eval EER 7.313%


  0%|          | 0/793 [00:00<?, ?it/s]

  0%|          | 0/2227 [00:00<?, ?it/s]

...

  0%|          | 0/793 [00:00<?, ?it/s]

  0%|          | 0/2227 [00:00<?, ?it/s]

epoch 12 | train loss 0.0033 | eval loss 0.8843 | eval EER 4.760%


epoch,▁▂▂▃▄▄▅▅▆▇▇█
eval_eer,██▄▃▃▆▆▄▃▁▁▂
eval_loss,▅▁▄▁▃▄█▁▃▄▃▆
train_loss,█▂▂▂▂▂▁▁▁▁▁▁
best_eer,4.10368
best_epoch,11
epoch,12
eval_eer,4.7595
eval_loss,0.88431
train_loss,0.00326


best epoch: 11
best eval EER: 4.104
